# Brain MRI Tumor Classification

## Notebook 3 — Pretrained CNN Training and Fine-Tuning


## Notebook Overview

This notebook trains three ImageNet-pretrained CNN backbones on the same four MRI classes: **MobileNetV2, EfficientNetB0, and ResNet50**. Each model follows the same two-stage process: first train a new four-class head while the pretrained backbone is frozen, then fine-tune the upper 20% of its layers at a lower learning rate. The training and validation splits come from Notebook 1's verified manifest.

We use validation results to select a checkpoint for each architecture. The **test split is not loaded or evaluated in this notebook**; it is reserved for the final model comparison notebook.

**Use limitation:** this is an educational machine-learning exercise. Predictions and confidence scores are not medical diagnoses or clinical recommendations.


## Notebook Plan

1. Connect Google Drive and locate the manifest and `Models` output folder.
2. Load verified train and validation rows. Keep test rows out of this notebook.
3. Build a common image input pipeline and each model's required preprocessing.
4. Define a consistent four-class classifier head and class weights.
5. For each backbone, load ImageNet weights, freeze it, and train the new head.
6. Unfreeze the upper 20% of the backbone layers and fine-tune with a lower learning rate.
7. Select the better checkpoint by validation loss; report validation accuracy, macro-F1, and per-class metrics.
8. Save only the selected validation checkpoint for each model plus its preprocessing metadata.


## Step 1 — Connect Drive and set paths

Use the same Drive `Datasets` folder as Notebooks 1 and 2. Notebook 1 saves `brain_mri_dataset_manifest.csv` directly inside this folder. The three selected model files and their metadata will be saved in a sibling `Models` folder.


In [2]:
!pip install tensorflow

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

DATASET_ROOT = Path('/content/drive/MyDrive/Brain MRI Tumor Classification/Datasets')
PROJECT_ROOT = DATASET_ROOT.parent
MANIFEST_PATH = DATASET_ROOT / 'brain_mri_dataset_manifest.csv'
MODEL_DIR = PROJECT_ROOT / 'Models'
CHECKPOINT_DIR = Path('/content/pretrained_cnn_checkpoints')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 12
FINE_TUNE_EPOCHS = 8
SEED = 42
CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
NUM_CLASSES = len(CLASS_NAMES)

tf.keras.utils.set_random_seed(SEED)
print('TensorFlow version:', tf.__version__)
print('Manifest:', MANIFEST_PATH)
print('Models will be saved to:', MODEL_DIR)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'Manifest not found: {MANIFEST_PATH}. Run Notebook 1 first.')


Mounted at /content/drive
TensorFlow version: 2.21.0
Manifest: /content/drive/MyDrive/Brain MRI Tumor Classification/Datasets/brain_mri_dataset_manifest.csv
Models will be saved to: /content/drive/MyDrive/Brain MRI Tumor Classification/Models
GPU devices: []


## Step 2 — Load verified train and validation rows

Only rows marked `label_verified` by Notebook 1 are used. We keep the test split out of the workflow so it cannot influence training, fine-tuning, or checkpoint selection.


In [4]:
manifest = pd.read_csv(MANIFEST_PATH)
required = {'filepath', 'split', 'label', 'label_verified'}
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f'Manifest is missing columns: {sorted(missing)}')

verified = manifest[
    manifest['label_verified'].astype(str).str.lower().isin(['true', '1'])
    & manifest['filepath'].notna()
    & manifest['label'].notna()
].copy()
verified['filepath'] = verified['filepath'].astype(str).str.strip()
verified['label'] = verified['label'].astype(str).str.strip()
unknown_labels = sorted(set(verified['label']) - set(CLASS_NAMES))
if unknown_labels:
    raise ValueError(f'Unexpected class labels: {unknown_labels}')

train_df = verified[verified['split'] == 'train'].copy()
valid_df = verified[verified['split'] == 'valid'].copy()
if train_df.empty or valid_df.empty:
    raise ValueError('Verified train and valid rows are both required.')

for split_name, frame in [('train', train_df), ('valid', valid_df)]:
    print(f'{split_name}: {len(frame):,} verified images')
    display(frame['label'].value_counts().reindex(CLASS_NAMES, fill_value=0).rename('images'))


train: 1,695 verified images


,images
label,
Glioma,564
Meningioma,358
No Tumor,335
Pituitary,438


valid: 502 verified images


,images
label,
Glioma,161
Meningioma,124
No Tumor,99
Pituitary,118


## Step 3 — Build shared image pipelines

Images are read from their saved paths, decoded as RGB, and resized to 224 × 224. Mild rotation and zoom are applied only to training images. Validation images are never augmented. Each pretrained architecture has its own required pixel preprocessing, which is applied after resizing and augmentation.


In [5]:
AUTOTUNE = tf.data.AUTOTUNE
LABEL_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03, seed=SEED),
    tf.keras.layers.RandomZoom(0.05, seed=SEED),
], name='training_augmentation')

def load_and_resize(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)  # float pixels in [0, 255]
    return image, label

def make_raw_dataset(frame, training=False):
    paths = frame['filepath'].astype(str).to_numpy()
    labels = frame['label'].map(LABEL_TO_ID).to_numpy(dtype=np.int32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return dataset.map(load_and_resize, num_parallel_calls=AUTOTUNE)

train_raw = make_raw_dataset(train_df, training=True)
valid_raw = make_raw_dataset(valid_df, training=False)

def make_model_dataset(raw_dataset, preprocessing_fn, training=False):
    def transform(image, label):
        if training:
            image = augmentation(image, training=True)
        image = preprocessing_fn(image)
        return image, label
    return raw_dataset.map(transform, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print('Shared train and validation image pipelines are ready.')


Shared train and validation image pipelines are ready.


### Model-specific preprocessing

Keras expects a different input transform for each backbone. MobileNetV2 scales pixels to −1…1; ResNet50 converts RGB to BGR and subtracts ImageNet channel means; EfficientNetB0 includes its rescaling layer in the model, so it receives resized float pixels in 0…255. We will keep these differences explicit rather than apply one normalization rule to every model.


In [6]:
PREPROCESSING = {
    'mobilenetv2': {
        'function': tf.keras.applications.mobilenet_v2.preprocess_input,
        'description': 'MobileNetV2 preprocess_input; scales RGB pixels to [-1, 1].',
    },
    'efficientnetb0': {
        'function': lambda image: image,
        'description': 'Pass float RGB pixels [0, 255]; EfficientNetB0 includes preprocessing.',
    },
    'resnet50': {
        'function': tf.keras.applications.resnet50.preprocess_input,
        'description': 'ResNet50 preprocess_input; RGB to BGR and ImageNet channel centering.',
    },
}


## Step 4 — Define the common classifier head and class weights

Each backbone feeds the same global-average-pooling, dropout, and four-class softmax head. The different backbones are the main comparison variable. Class weights are calculated from train counts so the model is not rewarded for favoring only the most common classes.


In [7]:
train_counts = train_df['label'].value_counts().reindex(CLASS_NAMES, fill_value=0)
if (train_counts == 0).any():
    raise ValueError(f'A class has no verified training images: {train_counts.to_dict()}')
class_weights = {
    LABEL_TO_ID[label]: float(len(train_df) / (NUM_CLASSES * train_counts[label]))
    for label in CLASS_NAMES
}
print('Training class weights:', class_weights)

def build_transfer_model(model_key, application_constructor):
    backbone = application_constructor(
        include_top=False, weights='imagenet',
        input_shape=(*IMAGE_SIZE, 3), pooling='avg',
    )
    # Keep Keras's default application name so it requests the official weights file.
    backbone_name = backbone.name
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='preprocessed_mri_image')
    features = backbone(inputs, training=False)
    features = tf.keras.layers.Dropout(0.30, name='classifier_dropout')(features)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', name='class_probabilities')(features)
    model = tf.keras.Model(inputs, outputs, name=f'{model_key}_four_class_classifier')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy', metrics=['accuracy'],
    )
    return model, backbone_name

MODEL_BUILDERS = {
    'mobilenetv2': tf.keras.applications.MobileNetV2,
    'efficientnetb0': tf.keras.applications.EfficientNetB0,
    'resnet50': tf.keras.applications.ResNet50,
}


Training class weights: {0: 0.7513297872340425, 1: 1.183659217877095, 2: 1.2649253731343284, 3: 0.9674657534246576}


## Step 5 — Train the new classifier head

For each architecture, Keras downloads its ImageNet pretrained weights if they are not already cached in the Colab runtime. The backbone stays frozen in this stage, so only the new MRI classifier head learns. Validation loss guides early stopping and saves the best head-training checkpoint.


The next code cell defines the reusable two-stage training procedure used for all three models.

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

def model_callbacks(checkpoint_path, patience):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path, monitor='val_loss', mode='min', save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', mode='min', patience=patience, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', mode='min', factor=0.3, patience=2, min_lr=1e-7, verbose=1
        ),
    ]

def evaluate_validation(model, validation_dataset, expected_count):
    loss, accuracy = model.evaluate(validation_dataset, verbose=0)
    probabilities = model.predict(validation_dataset, verbose=0)
    y_true = np.concatenate([labels.numpy() for _, labels in validation_dataset])
    if len(y_true) != expected_count:
        raise ValueError('Validation labels and predictions are not aligned.')
    y_pred = probabilities.argmax(axis=1)
    return {
        'val_loss': float(loss),
        'val_accuracy': float(accuracy),
        'val_macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'y_true': y_true, 'y_pred': y_pred, 'probabilities': probabilities,
    }

def train_one_backbone(model_key, application_constructor):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    preprocessing = PREPROCESSING[model_key]['function']
    train_ds = make_model_dataset(train_raw, preprocessing, training=True)
    valid_ds = make_model_dataset(valid_raw, preprocessing, training=False)

    print(f'\n===== {model_key.upper()} — frozen backbone stage =====')
    model, backbone_name = build_transfer_model(model_key, application_constructor)
    head_checkpoint = CHECKPOINT_DIR / f'{model_key}_head_best.keras'
    head_history = model.fit(
        train_ds, validation_data=valid_ds, epochs=HEAD_EPOCHS,
        class_weight=class_weights, callbacks=model_callbacks(head_checkpoint, patience=4),
    )
    head_model = tf.keras.models.load_model(head_checkpoint)
    head_metrics = evaluate_validation(head_model, valid_ds, len(valid_df))

    print(f'\n===== {model_key.upper()} — upper-layer fine-tuning stage =====')
    fine_tune_model = tf.keras.models.load_model(head_checkpoint)
    backbone = fine_tune_model.get_layer(backbone_name)
    backbone.trainable = True
    freeze_before = int(len(backbone.layers) * 0.80)
    for index, layer in enumerate(backbone.layers):
        layer.trainable = (index >= freeze_before and not isinstance(layer, tf.keras.layers.BatchNormalization))
    fine_tune_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy', metrics=['accuracy'],
    )
    fine_checkpoint = CHECKPOINT_DIR / f'{model_key}_fine_tuned_best.keras'
    fine_history = fine_tune_model.fit(
        train_ds, validation_data=valid_ds, epochs=FINE_TUNE_EPOCHS,
        class_weight=class_weights, callbacks=model_callbacks(fine_checkpoint, patience=3),
    )
    fine_model = tf.keras.models.load_model(fine_checkpoint)
    fine_metrics = evaluate_validation(fine_model, valid_ds, len(valid_df))

    if fine_metrics['val_loss'] < head_metrics['val_loss']:
        selected_model, selected_metrics = fine_model, fine_metrics
        selected_stage, selected_checkpoint = 'fine_tuned', fine_checkpoint
    else:
        selected_model, selected_metrics = head_model, head_metrics
        selected_stage, selected_checkpoint = 'frozen_head', head_checkpoint

    final_path = MODEL_DIR / f'{model_key}_best_validation.keras'
    selected_model.save(final_path)
    report = classification_report(
        selected_metrics['y_true'], selected_metrics['y_pred'],
        labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    summary = {
        'model': model_key,
        'selected_stage': selected_stage,
        'selected_checkpoint_stage': selected_checkpoint.name,
        'selection_rule': 'lowest validation loss between frozen-head and fine-tuned checkpoints',
        'validation_loss': selected_metrics['val_loss'],
        'validation_accuracy': selected_metrics['val_accuracy'],
        'validation_macro_f1': selected_metrics['val_macro_f1'],
        'preprocessing': PREPROCESSING[model_key]['description'],
        'class_names_in_output_order': CLASS_NAMES,
        'model_path': str(final_path),
        'per_class_report': {name: report[name] for name in CLASS_NAMES},
    }
    with open(MODEL_DIR / f'{model_key}_validation_summary.json', 'w', encoding='utf-8') as file:
        json.dump(summary, file, indent=2)

    print('Selected stage:', selected_stage)
    print('Validation loss / accuracy / macro-F1:',
          round(summary['validation_loss'], 4),
          round(summary['validation_accuracy'], 4),
          round(summary['validation_macro_f1'], 4))
    print(classification_report(
        selected_metrics['y_true'], selected_metrics['y_pred'],
        labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES, digits=3, zero_division=0,
    ))
    result = {key: value for key, value in summary.items() if key != 'per_class_report'}
    del model, head_model, fine_tune_model, fine_model, selected_model, backbone, train_ds, valid_ds
    gc.collect()
    tf.keras.backend.clear_session()
    return result


## Step 6 — Train MobileNetV2, EfficientNetB0, and ResNet50

This cell runs the same frozen-head and upper-layer fine-tuning procedure for each model. It writes one selected validation checkpoint per architecture to `Models`. The two intermediate stage checkpoints stay in Colab's temporary `/content` folder, not in the Drive `Models` folder.

The first run downloads ImageNet weights; internet access is needed in Colab. Later runs may reuse the Colab runtime's cached weights.


In [9]:
results = []
for model_key, constructor in MODEL_BUILDERS.items():
    saved_model = MODEL_DIR / f'{model_key}_best_validation.keras'
    saved_summary = MODEL_DIR / f'{model_key}_validation_summary.json'
    if saved_model.is_file() and saved_summary.is_file():
        print(f'Skipping {model_key}: a completed validation-selected model already exists.')
        with open(saved_summary, 'r', encoding='utf-8') as file:
            results.append(json.load(file))
    else:
        results.append(train_one_backbone(model_key, constructor))

validation_comparison = (
    pd.DataFrame(results)
      .set_index('model')
      .sort_values('validation_loss')
)
display(validation_comparison[['selected_stage', 'validation_loss',
                                'validation_accuracy', 'validation_macro_f1',
                                'model_path']])


Skipping mobilenetv2: a completed validation-selected model already exists.
Skipping efficientnetb0: a completed validation-selected model already exists.

===== RESNET50 — frozen backbone stage =====
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/12
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.4102 - loss: 1.4670
Epoch 1: val_loss improved from None to 0.54444, saving model to /content/pretrained_cnn_checkpoints/resnet50_head_best.keras

Epoch 1: finished saving model to /content/pretrained_cnn_checkpoints/resnet50_head_best.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 582s 11s/step - accuracy: 0.5687 - loss: 1.0726 - val_accuracy: 0.8028 - val_loss: 0.5444 - learning_rate: 0.0010
Epoch 2/12
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.8184 - loss: 0.5286
Epoch 2: val_loss improved from 0.54444 to 0.40435, saving model to /content/pretrained_cnn_checkpoints/resnet50_head_best.keras

Epoch 2: finished saving model to /content/pretrained_cnn_checkpoints/resnet50_head_best.

,selected_stage,validation_loss,validation_accuracy,validation_macro_f1,model_path
model,,,,,
resnet50,fine_tuned,0.218807,0.942231,0.939285,/content/drive/MyDrive/Brain MRI Tumor Classif...
mobilenetv2,fine_tuned,0.359147,0.886454,0.881700,/content/drive/MyDrive/Brain MRI Tumor Classif...
efficientnetb0,fine_tuned,0.419707,0.838645,0.831353,/content/drive/MyDrive/Brain MRI Tumor Classif...


## Step 7 — Save a shared model and preprocessing index

The three selected models are ready for the final comparison notebook. This index records each model file, its preprocessing rule, and the common output label order. No test metrics are included because the test split has not been used here.


In [10]:
model_index = {
    'class_names_in_output_order': CLASS_NAMES,
    'image_size': list(IMAGE_SIZE),
    'models': {
        row['model']: {
            'model_path': row['model_path'],
            'preprocessing': row['preprocessing'],
            'selected_stage': row['selected_stage'],
            'validation_loss': row['validation_loss'],
            'validation_accuracy': row['validation_accuracy'],
            'validation_macro_f1': row['validation_macro_f1'],
        }
        for row in results
    },
    'test_split_used': False,
}
index_path = MODEL_DIR / 'pretrained_model_index.json'
with open(index_path, 'w', encoding='utf-8') as file:
    json.dump(model_index, file, indent=2)
print('Saved model index:', index_path)
print('Saved model files:')
for row in results:
    print(' -', row['model_path'])


Saved model index: /content/drive/MyDrive/Brain MRI Tumor Classification/Models/pretrained_model_index.json
Saved model files:
 - /content/drive/MyDrive/Brain MRI Tumor Classification/Models/mobilenetv2_best_validation.keras
 - /content/drive/MyDrive/Brain MRI Tumor Classification/Models/efficientnetb0_best_validation.keras
 - /content/drive/MyDrive/Brain MRI Tumor Classification/Models/resnet50_best_validation.keras


## Notebook 3 complete

MobileNetV2, EfficientNetB0, and ResNet50 have each followed the same two-stage workflow. Their selected checkpoints are based on validation loss, with validation accuracy, macro-F1, and per-class results also recorded. Notebook 4 will load these three models plus the custom CNN from Notebook 2 and perform the final comparison on the held-out test split.
